# 🌸 Project 04 — Iris Flower Classifier

**Learning Goals**
- Understand what multi-class classification is
- Learn how to split data into training and test sets
- Train and compare two classic classifiers: **KNN** and **Decision Tree**
- Evaluate models with accuracy, classification report, and confusion matrix
- Visualise a decision boundary and a tree diagram

**Estimated Time:** 30–45 minutes

---

The **Iris dataset** is one of the most famous datasets in machine learning. It contains 150 measurements of iris flowers from three species:

| Species | Class Label |
|---------|-------------|
| *Iris setosa* | 0 |
| *Iris versicolor* | 1 |
| *Iris virginica* | 2 |

Each flower has **4 measurements** (features): sepal length, sepal width, petal length, petal width (all in cm).

## Step 1 — Import Libraries

We need `scikit-learn` for models and preprocessing, `matplotlib`/`seaborn` for plots, and `numpy` for array maths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.base import clone

print("✅ Libraries loaded successfully")

## Step 2 — Load and Explore the Dataset

Scikit-learn ships with the Iris dataset built-in — no download needed!

In [ ]:
iris = load_iris()

X = iris.data            # Feature matrix: shape (150, 4)
y = iris.target          # Label vector:   shape (150,)
feature_names = iris.feature_names
target_names  = iris.target_names

print(f"Feature matrix shape : {X.shape}")
print(f"Label vector shape   : {y.shape}")
print(f"Features : {list(feature_names)}")
print(f"Classes  : {list(target_names)}")
print(f"\nFirst 5 rows:")
print(X[:5])

### Visualise feature distributions

A **pairplot** shows every feature plotted against every other feature, coloured by class. It's a quick way to see whether classes are linearly separable.

In [ ]:
import pandas as pd

df = pd.DataFrame(X, columns=feature_names)
df['species'] = [target_names[i] for i in y]

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
colors = ['#e6194b', '#3cb44b', '#4363d8']
for ax, feat in zip(axes, feature_names):
    for cls_idx, cls_name in enumerate(target_names):
        subset = df[df['species'] == cls_name][feat]
        ax.hist(subset, bins=12, alpha=0.6, color=colors[cls_idx], label=cls_name)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('cm')
axes[0].legend(fontsize=8)
plt.suptitle('Feature Distributions by Species', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## Step 3 — Train / Test Split and Feature Scaling

We hold out **25%** of the data for evaluation. `stratify=y` ensures each split has the same class proportions.

**Why scale?** KNN calculates *distances* between points. Without scaling, a feature with large values (e.g. sepal length in cm) would dominate over a feature with small values. `StandardScaler` transforms each feature to have mean 0 and standard deviation 1.

> ⚠️ Always fit the scaler on **training data only**, then apply the same transform to test data. This prevents *data leakage*.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print(f"Training samples : {len(y_train)}")
print(f"Test samples     : {len(y_test)}")
print(f"\nBefore scaling (first row): {X_train[0]}")
print(f"After scaling  (first row): {X_train_scaled[0].round(3)}")

## Step 4 — Train a K-Nearest Neighbors (KNN) Classifier

**How KNN works:**  
1. Store all training examples.
2. For a new point, find the *k* nearest training examples (using Euclidean distance).
3. Predict the majority class among those *k* neighbours.

We start with **k = 5**.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)

print(f"KNN Accuracy: {acc_knn:.2%}\n")
print(classification_report(y_test, y_pred_knn, target_names=target_names))

## Step 5 — Train a Decision Tree Classifier

**How a Decision Tree works:**  
1. At each node, pick the feature + threshold that best splits the data (maximises information gain / Gini purity).
2. Recurse on each sub-group until a stopping criterion is met (`max_depth`).
3. Predict using the leaf node's majority class.

Decision Trees are **interpretable** — you can print the rules and explain every prediction.

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
acc_dt = accuracy_score(y_test, y_pred_dt)

print(f"Decision Tree Accuracy: {acc_dt:.2%}\n")
print(classification_report(y_test, y_pred_dt, target_names=target_names))

## Step 6 — Confusion Matrices

A **confusion matrix** shows how many samples of each true class were predicted as each class.  
- The **diagonal** = correct predictions.  
- **Off-diagonal** cells = mistakes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_knn, y_pred_dt],
    [f'KNN (k=5) — Accuracy {acc_knn:.2%}', f'Decision Tree — Accuracy {acc_dt:.2%}']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 7 — Visualise the Decision Tree

One of the best features of Decision Trees: you can actually *see* every rule the model learned!

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    dt,
    feature_names=feature_names,
    class_names=target_names,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title("Decision Tree — Learned Rules (max_depth=4)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 8 — Decision Boundary Plot (2 Features)

We re-train each model using only the **first two features** (sepal length & sepal width) so we can plot the decision regions in 2D. This gives an intuitive picture of how each algorithm "carves up" the feature space.

In [ ]:
X_all_scaled = np.vstack([X_train_scaled, X_test_scaled])
y_all        = np.concatenate([y_train, y_test])
X2 = X_all_scaled[:, :2]  # sepal length + sepal width only

h = 0.02
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e6194b', '#3cb44b', '#4363d8']

for ax, model, label in zip(axes, [knn, dt], ['KNN (k=5)', 'Decision Tree']):
    model_2d = clone(model)
    model_2d.fit(X2, y_all)
    Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.RdYlBu)
    for cls_idx, cls_name in enumerate(target_names):
        mask = y_all == cls_idx
        ax.scatter(X2[mask, 0], X2[mask, 1], c=colors[cls_idx],
                   label=cls_name, edgecolors='k', s=40, alpha=0.8)
    ax.set_xlabel(f"{feature_names[0]} (scaled)")
    ax.set_ylabel(f"{feature_names[1]} (scaled)")
    ax.set_title(f"{label} — Decision Boundary", fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Step 9 — Predict a New Flower

Let's predict the species of a flower we haven't seen before.

In [ ]:
# sepal length, sepal width, petal length, petal width (cm)
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])   # small petals → likely Setosa

new_scaled = scaler.transform(new_flower)

knn_pred = target_names[knn.predict(new_scaled)[0]]
dt_pred  = target_names[dt.predict(new_scaled)[0]]

# KNN also gives probability-like confidence via vote counts
knn_proba = knn.predict_proba(new_scaled)[0]
dt_proba  = dt.predict_proba(new_scaled)[0]

print(f"New flower measurements: {new_flower[0]}")
print()
print(f"KNN prediction    : {knn_pred}")
print(f"  Confidence      : {dict(zip(target_names, knn_proba.round(2)))}")
print()
print(f"DT  prediction    : {dt_pred}")
print(f"  Confidence      : {dict(zip(target_names, dt_proba.round(2)))}")

## Step 10 — Try It Yourself! 🧪

Experiment with the code below to deepen your understanding:

1. **Change k in KNN** — try `n_neighbors=1`, `3`, `10`, `20`. How does accuracy change? Why does very small k overfit?
2. **Change max_depth** — try `max_depth=1`, `2`, `10`, `None`. What happens to the tree and accuracy?
3. **Use only 2 features** — re-train both models with only `petal length` and `petal width` (columns 2 & 3). Is accuracy higher or lower? Why?
4. **Try your own flower** — change `new_flower` measurements and see which species both models predict.

In [ ]:
# ---- Experiment: change k ----
k_values = [1, 3, 5, 7, 10, 15, 20]
accs = []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    accs.append(accuracy_score(y_test, m.predict(X_test_scaled)))

plt.figure(figsize=(7, 4))
plt.plot(k_values, accs, marker='o', color='steelblue')
plt.xlabel('k (number of neighbours)')
plt.ylabel('Test Accuracy')
plt.title('KNN Accuracy vs k')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for k, a in zip(k_values, accs):
    print(f"  k={k:2d}  →  accuracy={a:.2%}")

## Summary

| Concept | Plain English Explanation |
|---------|---------------------------|
| Multi-class classification | Predict which of 3+ categories a sample belongs to |
| Train/test split | Separate data so we evaluate on unseen examples |
| Stratified split | Keep the same class proportions in train and test |
| StandardScaler | Rescale features to mean=0, std=1 so distances are fair |
| KNN (k=5) | Classify by majority vote of the 5 closest training points |
| Decision Tree | Learn a hierarchy of if/else rules from training data |
| max_depth | Limit tree depth to prevent overfitting |
| Confusion matrix | Table showing true vs predicted class counts |
| Precision | Of all samples predicted as class X, how many really were X? |
| Recall | Of all real class X samples, how many did we correctly find? |
| F1-score | Harmonic mean of precision and recall (balanced metric) |

---
*Part of the **Daily AI Projects** series — one beginner ML project per weekday.*